In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

RUN_MODELS = ["distilbert-base-uncased", "bert-base-uncased"]
RUN_TASKS  = ["sst2", "cola", "mrpc"]
SEEDS = None                     # None = every checkpoint found; or e.g. [0, 1, 2]
K_LAYER = [8, 6, 4, 2, 0]          # heads kept in the pruned layer (0 = whole attention block removed)
METHODS_L = ["random", "frob", "op", "chain", "grad"]
N_RANDOM = 3                       # random draws per (layer, k)
GRAD_N = 512                       # dev examples for gradient importance
OUT_DIR = "/kaggle/working/layerwise_results"
SMOKE = False                      # True: 1 checkpoint, 2 layers, K=[4,0], 1 random draw (sanity check only)
SEARCH = [
    "/kaggle/input/notebooks/*/*/ckpt/*/done.json",
    "/kaggle/input/*/ckpt/*/done.json",
    "/kaggle/input/**/done.json",
    "/kaggle/working/ckpt/**/done.json",
]
MAX_LEN = 128

import glob, json, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

if SMOKE:
    K_LAYER, N_RANDOM, OUT_DIR = [4, 0], 1, OUT_DIR + "_smoke"


def keep_layer_only(score_row, layer, k, L, H):
    """Mask that prunes ONLY `layer` to its k best heads by score_row (length H); all other layers dense."""
    keep = np.ones((L, H), bool)
    keep[layer, :] = False
    if k > 0:
        keep[layer, np.argsort(-score_row, kind="stable")[:k]] = True
    return keep


import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import matthews_corrcoef, f1_score

dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEYS = {"sst2": ("sentence", None), "cola": ("sentence", None), "mrpc": ("sentence1", "sentence2")}


def get_attn(model):
    if hasattr(model, "distilbert"):
        ls = model.distilbert.transformer.layer
        return [(l.attention.q_lin, l.attention.k_lin, l.attention.v_lin, l.attention.out_lin) for l in ls], \
            model.config.n_heads, model.config.dim
    ls = model.bert.encoder.layer
    return [(l.attention.self.query, l.attention.self.key, l.attention.self.value, l.attention.output.dense) for l in ls], \
        model.config.num_attention_heads, model.config.hidden_size


def prep(task, tok):
    ds = load_dataset("nyu-mll/glue", task)
    k1, k2 = KEYS[task]

    def f(b):
        return tok(b[k1], b[k2], truncation=True, max_length=MAX_LEN) if k2 else tok(b[k1], truncation=True, max_length=MAX_LEN)

    split = ds["train"].train_test_split(test_size=0.05, seed=0)           # identical to fine-tuning script
    parts = {"dev": split["test"], "val": ds["validation"]}
    cols = ["input_ids", "attention_mask", "label"]
    out = {}
    for n, d in parts.items():
        d = d.map(f, batched=True)
        out[n] = d.remove_columns([c for c in d.column_names if c not in cols])
    out["dev_grad"] = out["dev"].select(range(min(GRAD_N, len(out["dev"]))))
    return out


def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b["label"] for b in batch])
        enc = tok.pad([{k: b[k] for k in ("input_ids", "attention_mask")} for b in batch], return_tensors="pt")
        enc["labels"] = labels
        return enc
    return collate


@torch.no_grad()
def predict(model, ds, collate, bs=128):
    model.eval()
    lg, lb = [], []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        lab = b.pop("labels")
        b = {k: v.to(dev_) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            lg.append(model(**b).logits.float().cpu())
        lb.append(lab)
    return torch.cat(lg).numpy(), torch.cat(lb).numpy()


def metrics(task, logits, labels):
    p = logits.argmax(1)
    loss = torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()
    m = {"acc": float((p == labels).mean()), "loss": loss, "mcc": np.nan, "f1": np.nan}
    if task == "cola": m["mcc"] = float(matthews_corrcoef(labels, p))
    if task == "mrpc": m["f1"] = float(f1_score(labels, p, zero_division=0))
    m["primary"] = {"sst2": m["acc"], "cola": m["mcc"], "mrpc": m["f1"]}[task]
    return m


def head_scores(model):
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    S = {k: np.zeros((L, H)) for k in ["frob", "op", "chain", "kappa"]}
    with torch.no_grad():
        for l, (q, k, v, o) in enumerate(acc):
            Wq, Wk, Wv, Wo = [m.weight.detach().float().cpu() for m in (q, k, v, o)]
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                Woh = Wo[:, sl]
                sv = torch.linalg.svdvals(Woh)
                S["op"][l, h] = sv[0].item()
                S["frob"][l, h] = torch.linalg.matrix_norm(Woh, "fro").item()
                S["kappa"][l, h] = (sv[0] ** 2 / (sv ** 2).mean()).item()
                bq = torch.linalg.matrix_norm(Wq[sl, :], 2).item()
                bk = torch.linalg.matrix_norm(Wk[sl, :], 2).item()
                bv = torch.linalg.matrix_norm(Wv[sl, :], 2).item()
                S["chain"][l, h] = bq * bk * bv * sv[0].item()
    return S


def grad_importance(model, ds, collate):
    """Michel et al. (2019)-style: mean |dL/dxi_h|, xi = per-head gate on the out-projection input."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    xi = torch.ones(L, H, device=dev_, requires_grad=True)
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0]
            B, T, Dm = x.shape
            return ((x.reshape(B, T, H, dh) * xi[l].view(1, 1, H, 1)).reshape(B, T, Dm),)
        hooks.append(o.register_forward_pre_hook(pre))
    imp = torch.zeros(L, H, device=dev_)
    model.eval()
    for b in DataLoader(ds, batch_size=1, collate_fn=collate):
        b = {k: v.to(dev_) for k, v in b.items()}
        loss = model(**b).loss
        g, = torch.autograd.grad(loss, xi)
        imp += g.abs()
    for h_ in hooks:
        h_.remove()
    return (imp / len(ds)).detach().cpu().numpy()


class Pruner:
    def __init__(self, model):
        self.model = model
        self.acc, self.H, self.D = get_attn(model)
        self.dh = self.D // self.H
        self.W0 = [o.weight.detach().clone() for (_, _, _, o) in self.acc]

    def apply(self, keep):
        with torch.no_grad():
            for l, (_, _, _, o) in enumerate(self.acc):
                o.weight.copy_(self.W0[l])
                for h in range(self.H):
                    if not keep[l, h]:
                        o.weight[:, h * self.dh:(h + 1) * self.dh] = 0.0


def discover():
    runs = {}
    for pat in SEARCH:
        for p in glob.glob(pat, recursive=True):
            try:
                r = json.load(open(p))
            except Exception:
                continue
            runs[(r["model"], r["task"], r["seed"])] = os.path.dirname(p)
    sel = {k: v for k, v in runs.items() if k[0] in RUN_MODELS and k[1] in RUN_TASKS and (SEEDS is None or k[2] in SEEDS)}
    if not sel:
        print("No matching checkpoints. done.json files seen:", sorted(runs.keys()))
    sel = dict(sorted(sel.items()))
    if SMOKE:
        sel = dict(list(sel.items())[:1])
    return sel


def run_checkpoint(key, path, cache):
    mname, task, seed = key
    tag = f"{mname}_{task}_seed{seed}"
    odir = f"{OUT_DIR}/{tag}"
    if os.path.exists(f"{odir}/DONE"):
        print("skip (done):", tag); return
    os.makedirs(odir, exist_ok=True)
    if (mname, task) not in cache:
        tok = AutoTokenizer.from_pretrained(path)
        cache[(mname, task)] = (tok, prep(task, tok), make_collate(tok))
    tok, data, collate = cache[(mname, task)]
    model = AutoModelForSequenceClassification.from_pretrained(path).to(dev_).eval()
    pr = Pruner(model)
    L, H = len(pr.acc), pr.H
    layers = list(range(2)) if SMOKE else list(range(L))
    print(f"\n=== {tag}: L={L} H={H}  val n={len(data['val'])}")

    S = head_scores(model)
    G = grad_importance(model, data["dev_grad"], collate)
    scores = {"frob": S["frob"], "op": S["op"], "chain": S["chain"], "grad": G}
    layer_kappa, layer_op = S["kappa"].mean(1), S["op"].mean(1)

    rows, preds = [], {}
    t0 = time.time()

    def evaluate(layer, k, method, draw, keep):
        pr.apply(keep)
        lv, yv = predict(model, data["val"], collate)
        m = metrics(task, lv, yv)
        rows.append(dict(model=mname, task=task, seed=seed, layer=layer, k=k, method=method, draw=draw,
                         n_kept_layer=int(keep[layer].sum()) if layer >= 0 else H,
                         val_acc=m["acc"], val_loss=m["loss"], val_mcc=m["mcc"], val_f1=m["f1"], primary=m["primary"],
                         layer_mean_kappa=float(layer_kappa[layer]) if layer >= 0 else np.nan,
                         layer_mean_op=float(layer_op[layer]) if layer >= 0 else np.nan))
        preds[f"{method}|{layer}|{k}|{draw}"] = lv.argmax(1).astype(np.int8)
        if "labels" not in preds: preds["labels"] = yv.astype(np.int8)
        return m["primary"]

    dense = evaluate(-1, H, "dense", 0, np.ones((L, H), bool))
    print(f"  dense primary metric = {dense:.4f}")
    for layer in layers:
        for k in K_LAYER:
            if k == 0:
                evaluate(layer, 0, "remove_all", 0, keep_layer_only(np.zeros(H), layer, 0, L, H))
                continue
            for method in METHODS_L:
                if method == "random":
                    for d in range(N_RANDOM):
                        rs = np.random.RandomState(100000 * seed + 1000 * layer + 100 * k + d)
                        evaluate(layer, k, "random", d, keep_layer_only(rs.rand(H), layer, k, L, H))
                else:
                    evaluate(layer, k, method, 0, keep_layer_only(scores[method][layer], layer, k, L, H))
        print(f"  layer {layer} done  ({(time.time() - t0) / 60:.1f} min elapsed)")

    df = pd.DataFrame(rows)
    df["dense_primary"] = dense
    df["drop"] = dense - df["primary"]                       # positive = pruning hurt
    df.to_csv(f"{odir}/results.csv", index=False)
    np.savez_compressed(f"{odir}/val_preds.npz", **preds)
    np.savez_compressed(f"{odir}/head_scores.npz", **S, grad=G)

    # --- Q2 quick look: does layer mean-kappa predict layer sensitivity (drop when whole block removed)? ---
    d0 = df[(df.k == 0) & (df.layer >= 0)].sort_values("layer")
    if len(d0) >= 4:
        rho, p = spearmanr(d0.layer_mean_kappa, d0["drop"])
        print(f"  Q2: Spearman(layer mean kappa, drop at k=0) = {rho:.3f} (p={p:.3f}, n_layers={len(d0)})")
    json.dump(dict(tag=tag, dense_primary=dense, K_LAYER=K_LAYER, N_RANDOM=N_RANDOM, layers=layers,
                   minutes=(time.time() - t0) / 60), open(f"{odir}/meta.json", "w"), indent=2)
    open(f"{odir}/DONE", "w").write("ok")
    del model, pr
    torch.cuda.empty_cache()


def main():
    runs = discover()
    print(f"Found {len(runs)} checkpoints:")
    for k, v in runs.items():
        print("  ", k, "->", v)
    assert runs, "No checkpoints found: Add Input -> Notebook Output Files (fine-tuning notebooks)"
    cache = {}
    for key, path in runs.items():
        run_checkpoint(key, path, cache)
    allr = [pd.read_csv(p) for p in glob.glob(f"{OUT_DIR}/*/results.csv")]
    if allr:
        A = pd.concat(allr)
        A.to_csv(f"{OUT_DIR}/ALL_layerwise_results.csv", index=False)
        S_ = (A[A.layer >= 0].groupby(["model", "task", "layer", "k", "method"])["drop"].mean().reset_index())
        S_.to_csv(f"{OUT_DIR}/ALL_layerwise_summary.csv", index=False)
        print("\nMean drop (primary metric, points x100) per layer, k=0 (whole attention block removed):")
        print((S_[S_.k == 0].pivot_table(index=["model", "task"], columns="layer", values="drop") * 100).round(1).to_string())
    print("\nALL DONE ->", OUT_DIR)


if __name__ == "__main__" or True:
    main()

Found 18 checkpoints:
   ('bert-base-uncased', 'sst2', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed0
   ('bert-base-uncased', 'sst2', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed1
   ('bert-base-uncased', 'sst2', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed2
   ('distilbert-base-uncased', 'cola', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed0
   ('distilbert-base-uncased', 'cola', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed1
   ('distilbert-base-uncased', 'cola', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed2
   ('distilbert-base-uncased', 'cola', 3) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed3
   ('distilbert-base-uncased', 'cola', 4) -> /ka

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (63 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_sst2_seed3: L=6 H=12  val n=872
  dense primary metric = 0.9117
  layer 0 done  (0.2 min elapsed)
  layer 1 done  (0.3 min elapsed)
  layer 2 done  (0.5 min elapsed)
  layer 3 done  (0.6 min elapsed)
  layer 4 done  (0.8 min elapsed)
  layer 5 done  (1.0 min elapsed)
  Q2: Spearman(layer mean kappa, drop at k=0) = -0.580 (p=0.228, n_layers=6)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


=== distilbert-base-uncased_sst2_seed4: L=6 H=12  val n=872
  dense primary metric = 0.9094
  layer 0 done  (0.2 min elapsed)
  layer 1 done  (0.3 min elapsed)
  layer 2 done  (0.5 min elapsed)
  layer 3 done  (0.6 min elapsed)
  layer 4 done  (0.8 min elapsed)
  layer 5 done  (1.0 min elapsed)
  Q2: Spearman(layer mean kappa, drop at k=0) = -0.714 (p=0.111, n_layers=6)

Mean drop (primary metric, points x100) per layer, k=0 (whole attention block removed):
layer                          0    1    2     3     4     5    6    7    8    9    10   11
model                   task                                                               
bert-base-uncased       sst2 -0.2 -0.3  0.2   0.3  -0.2   0.2  0.3  0.6  0.3  0.5  0.2 -0.0
distilbert-base-uncased cola  6.8  8.1  7.8  10.0   5.4  16.6  NaN  NaN  NaN  NaN  NaN  NaN
                        mrpc  3.0  2.9  2.0  76.0  15.5  38.6  NaN  NaN  NaN  NaN  NaN  NaN
                        sst2  0.4  0.2 -0.3   3.5   3.1   3.5  NaN  NaN  NaN 